# OceanWatch Analytics · Entrega 2

## Carga a bronze y AutoLoad

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "oceanwatch"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
LANDING = f"{BASE}/landing"
CHECKPOINT = f"{BASE}/_checkpoints/bronze_ais"
SCHEMA_LOCATION = f"{BASE}/_schemas/bronze_ais"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_ais"

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}"
)

print(f"Landing: {LANDING}")
print(f"Checkpoint: {CHECKPOINT}")
print(f"Destino: {BRONZE_TABLE}")
print(f"Schema Location: {SCHEMA_LOCATION}")

Define el auto-loader:

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType

AIS_SCHEMA = StructType([
    StructField("MMSI",             StringType(),    nullable=True),
    StructField("BaseDateTime",     TimestampType(), nullable=True),
    StructField("LAT",              DoubleType(),    nullable=True),
    StructField("LON",              DoubleType(),    nullable=True),
    StructField("SOG",              DoubleType(),    nullable=True),
    StructField("COG",              DoubleType(),    nullable=True),
    StructField("Heading",          DoubleType(),    nullable=True),
    StructField("VesselName",       StringType(),    nullable=True),
    StructField("IMO",              StringType(),    nullable=True),
    StructField("CallSign",         StringType(),    nullable=True),
    StructField("VesselType",       IntegerType(),   nullable=True),
    StructField("Status",           IntegerType(),   nullable=True),
    StructField("Length",           DoubleType(),    nullable=True),
    StructField("Width",            DoubleType(),    nullable=True),
    StructField("Draft",            DoubleType(),    nullable=True),
    StructField("Cargo",            IntegerType(),   nullable=True),
    StructField("TransceiverClass", StringType(),    nullable=True),
    StructField("_corrupt_record",  StringType(),    nullable=True),
])

In [0]:
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .schema(AIS_SCHEMA)
    .load(LANDING)
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn(
        "source_modified_at",
        F.col("_metadata.file_modification_time")
    )
    .withColumn("ingestion_time", F.current_timestamp())
)

Revisa si hay archivos pendientes de procesar en LANDING y, si los encuentra, carga sus registros en la tabla Bronze

In [0]:
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()